In [3]:
!pip install -q yfinance plotly tabulate

import requests
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

import plotly.graph_objects as go
import plotly.express as px
import yfinance as yf

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [6]:
# --- EDIT THESE FOR YOUR ANALYSIS ---
TARGET_TICKER = "EPAM"                       # company you're valuing
COMPS_TICKERS = ["ACN", "CTSH", "DXC", "IBM"]  # 3-5 comparable companies

# SEC requires a descriptive User-Agent identifying you — use your real name/email
SEC_USER_AGENT = "Andrew Lim hslim3491@gmail.com"

ALL_TICKERS = [TARGET_TICKER] + COMPS_TICKERS

In [7]:
def get_cik_map():
    """Downloads SEC's full ticker->CIK lookup table."""
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": SEC_USER_AGENT}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return {v["ticker"].upper(): str(v["cik_str"]).zfill(10) for v in data.values()}

CIK_MAP = get_cik_map()

def ticker_to_cik(ticker):
    cik = CIK_MAP.get(ticker.upper())
    if cik is None:
        raise ValueError(f"CIK not found for ticker '{ticker}' — check spelling/listing.")
    return cik

print(f"CIK lookup loaded: {len(CIK_MAP):,} tickers available.")

CIK lookup loaded: 10,414 tickers available.


In [32]:
SEC_TAGS = {
    "Revenues": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet"],
    "CostOfRevenue": ["CostOfRevenue", "CostOfGoodsAndServicesSold"],
    "OperatingIncomeLoss": [
        "OperatingIncomeLoss",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
    ],
    "NetIncomeLoss": ["NetIncomeLoss"],
    "Assets": ["Assets"],
    "AssetsCurrent": ["AssetsCurrent"],
    "LiabilitiesCurrent": ["LiabilitiesCurrent"],
    "Liabilities": ["Liabilities"],
    "StockholdersEquity": ["StockholdersEquity"],
    "RetainedEarningsAccumulatedDeficit": ["RetainedEarningsAccumulatedDeficit", "RetainedEarningsUnappropriated"],
    "LongTermDebtNoncurrent": ["LongTermDebtNoncurrent"],
    "LongTermDebtCurrent": ["LongTermDebtCurrent"],
    "DepreciationDepletionAndAmortization": [
        "DepreciationDepletionAndAmortization",
        "DepreciationAmortizationAndAccretionNet",
    ],
    "CashAndCashEquivalentsAtCarryingValue": ["CashAndCashEquivalentsAtCarryingValue"],
    "NetCashProvidedByUsedInOperatingActivities": ["NetCashProvidedByUsedInOperatingActivities"],
    "CommonStockSharesOutstanding": ["CommonStockSharesOutstanding"],
}

def fetch_companyfacts(cik):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    headers = {"User-Agent": SEC_USER_AGENT}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()

def extract_annual_series(facts_json, tag_candidates):
    """
    Merges ALL candidate tags (not just the first match) since companies
    often switch XBRL tag names mid-history (e.g., EPAM switched from
    'Revenues' to 'RevenueFromContractWithCustomerExcludingAssessedTax'
    after adopting ASC 606 in 2018). combine_first fills gaps from
    earlier-priority tags using later candidates, covering the full history.

    Returns a Series indexed by period END DATE (not 'fy', which is unreliable
    since 10-Ks also tag quarterly comparison data under form='10-K').
    Flow items (income/cash flow) are filtered to ~330-400 day periods to
    keep only true annual figures. Instant items (balance sheet) need no
    duration filter since they're point-in-time.
    """
    us_gaap = facts_json.get("facts", {}).get("us-gaap", {})
    combined = pd.Series(dtype=float)
    for tag in tag_candidates:
        if tag not in us_gaap:
            continue
        units = us_gaap[tag].get("units", {}).get("USD", [])
        if not units:
            continue
        df = pd.DataFrame(units)
        df = df[df["form"] == "10-K"].copy()
        if df.empty:
            continue
        df["end"] = pd.to_datetime(df["end"])
        if "start" in df.columns and df["start"].notna().any():
            df["start"] = pd.to_datetime(df["start"])
            df["duration_days"] = (df["end"] - df["start"]).dt.days
            df = df[(df["duration_days"] >= 330) & (df["duration_days"] <= 400)]
        if df.empty:
            continue
        df = df.sort_values("end").drop_duplicates(subset="end", keep="last")
        series = df.set_index("end")["val"]
        combined = combined.combine_first(series)
    return combined.sort_index()

def build_financials_panel(ticker):
    try:
        cik = ticker_to_cik(ticker)
        facts = fetch_companyfacts(cik)
    except Exception as e:
        raise RuntimeError(f"Failed to fetch SEC data for {ticker}: {e}")

    data = {}
    for clean_name, candidates in SEC_TAGS.items():
        data[clean_name] = extract_annual_series(facts, candidates)

    panel = pd.DataFrame(data)
    panel.index = pd.to_datetime(panel.index)
    panel.index.name = "end"
    panel = panel.sort_index().tail(5)
    panel["fy"] = panel.index.year

    # Fallback: some filers (e.g. ACN) don't tag total "Liabilities" directly —
    # derive it from the balance sheet identity, which always holds:
    # Assets = Liabilities + StockholdersEquity
    panel["Liabilities"] = panel["Liabilities"].fillna(panel["Assets"] - panel["StockholdersEquity"])

    panel["ticker"] = ticker
    return panel.reset_index()

all_panels = {}
for tkr in ALL_TICKERS:
    try:
        all_panels[tkr] = build_financials_panel(tkr)
        print(f"✓ {tkr}: {len(all_panels[tkr])} years pulled")
    except Exception as e:
        print(f"✗ {tkr}: FAILED — {e}")
    time.sleep(0.3)

if TARGET_TICKER not in all_panels:
    raise SystemExit(f"Target ticker {TARGET_TICKER} failed to load — cannot continue.")

✓ EPAM: 5 years pulled
✓ ACN: 5 years pulled
✓ CTSH: 5 years pulled
✓ DXC: 5 years pulled
✓ IBM: 5 years pulled


In [33]:
def compute_ratios(df):
    df = df.copy()
    df["GrossProfit"] = df["Revenues"] - df["CostOfRevenue"]
    df["GrossMargin"] = df["GrossProfit"] / df["Revenues"]
    df["OperatingMargin"] = df["OperatingIncomeLoss"] / df["Revenues"]
    df["NetMargin"] = df["NetIncomeLoss"] / df["Revenues"]
    df["ROE"] = df["NetIncomeLoss"] / df["StockholdersEquity"]
    df["AssetTurnover"] = df["Revenues"] / df["Assets"]
    df["EquityMultiplier"] = df["Assets"] / df["StockholdersEquity"]
    df["ROA"] = df["NetIncomeLoss"] / df["Assets"]
    df["CurrentRatio"] = df["AssetsCurrent"] / df["LiabilitiesCurrent"]
    df["TotalDebt"] = df["LongTermDebtNoncurrent"].fillna(0) + df["LongTermDebtCurrent"].fillna(0)
    df["EBITDA"] = df["OperatingIncomeLoss"] + df["DepreciationDepletionAndAmortization"].fillna(0)
    df["DebtToEBITDA"] = df["TotalDebt"] / df["EBITDA"]
    return df

def flag_yoy_anomalies(df, col, threshold=2.0):
    pct_change = df[col].pct_change()
    z = (pct_change - pct_change.mean()) / pct_change.std()
    return z.abs() > threshold

ratios_by_ticker = {t: compute_ratios(p) for t, p in all_panels.items()}
target_ratios = ratios_by_ticker[TARGET_TICKER]
target_ratios["NetMargin_anomaly"] = flag_yoy_anomalies(target_ratios, "NetMargin")

target_ratios[["fy", "Revenues", "GrossMargin", "OperatingMargin", "NetMargin", "ROE", "CurrentRatio", "DebtToEBITDA"]]

,fy,Revenues,GrossMargin,OperatingMargin,NetMargin,ROE,CurrentRatio,DebtToEBITDA
0,2021,"3,758,144,000.000",0.339,0.144,0.128,0.194,2.973,0.056
1,2022,"4,824,698,000.000",0.319,0.119,0.087,0.140,3.692,0.048
2,2023,"4,690,540,000.000",0.306,0.107,0.089,0.120,4.793,0.052
3,2024,"4,727,940,000.000",0.307,0.115,0.096,0.125,2.955,0.046
4,2025,"5,457,056,000.000",0.288,0.095,0.069,0.103,2.594,0.048


In [34]:
fig = go.Figure()
for metric in ["GrossMargin", "OperatingMargin", "NetMargin"]:
    fig.add_trace(go.Scatter(
        x=target_ratios["fy"], y=target_ratios[metric] * 100,
        mode="lines+markers", name=metric
    ))
fig.update_layout(
    title=f"{TARGET_TICKER} — Margin Trends (5-Year)",
    yaxis_title="Margin (%)", xaxis_title="Fiscal Year", template="plotly_white"
)
fig.show()

In [35]:
def get_market_data(ticker):
    t = yf.Ticker(ticker)
    info = t.info
    return {
        "ticker": ticker,
        "price": info.get("currentPrice") or info.get("regularMarketPrice"),
        "market_cap": info.get("marketCap"),
        "beta": info.get("beta") or 1.0,
        "trailing_pe": info.get("trailingPE"),
        "ev_to_ebitda_yf": info.get("enterpriseToEbitda"),
        "total_debt": info.get("totalDebt") or 0,
        "total_cash": info.get("totalCash") or 0,
    }

market_data = {}
for t in ALL_TICKERS:
    try:
        market_data[t] = get_market_data(t)
        print(f"✓ {t}: market cap ${market_data[t]['market_cap']:,.0f}")
    except Exception as e:
        print(f"✗ {t}: FAILED — {e}")

market_df = pd.DataFrame(market_data).T
market_df

✓ EPAM: market cap $3,958,562,048
✓ ACN: market cap $75,770,667,008
✓ CTSH: market cap $19,854,174,208
✓ DXC: market cap $1,342,834,944
✓ IBM: market cap $235,826,626,560


,ticker,price,market_cap,beta,trailing_pe,ev_to_ebitda_yf,total_debt,total_cash
EPAM,EPAM,75.770,3958562048,1.399,10.886,4.451,287943008,1036958976
ACN,ACN,123.820,75770667008,1.069,9.890,6.038,8388769792,10171567104
CTSH,CTSH,41.975,19854174208,0.810,9.105,5.213,1092000000,1516999936
DXC,DXC,8.285,1342834944,0.808,82.850,2.880,4247000064,1736999936
IBM,IBM,250.910,235826626560,0.665,22.185,17.592,69802000384,11783000064


In [36]:
def altman_z_score(df, market_cap_series):
    """
    Classic Altman Z-Score. Calibrated originally for manufacturers —
    treat zone labels as directional signals for services firms, not absolute thresholds.
    market_cap_series should ideally be historical mkt cap per fiscal year;
    here we approximate with current market cap (documented limitation — see Upgrades).
    """
    z = pd.DataFrame(index=df.index)
    working_capital = df["AssetsCurrent"] - df["LiabilitiesCurrent"]
    z["A"] = working_capital / df["Assets"]
    z["B"] = df["RetainedEarningsAccumulatedDeficit"] / df["Assets"]
    z["C"] = df["OperatingIncomeLoss"] / df["Assets"]
    z["D"] = market_cap_series / df["Liabilities"]
    z["E"] = df["Revenues"] / df["Assets"]
    z["Z_score"] = 1.2*z["A"] + 1.4*z["B"] + 3.3*z["C"] + 0.6*z["D"] + 1.0*z["E"]
    z["Zone"] = z["Z_score"].apply(lambda s: "Safe" if s > 2.99 else ("Grey" if s > 1.81 else "Distress"))
    return z

def piotroski_f_score(df):
    """9-point binary financial-strength score; first year is NaN (needs a prior year)."""
    df = df.reset_index(drop=True)
    scores = [None]
    for i in range(1, len(df)):
        prev, curr = df.iloc[i - 1], df.iloc[i]
        s = 0
        s += int(curr["ROA"] > 0)
        cfo = curr["NetCashProvidedByUsedInOperatingActivities"]
        s += int(cfo > 0)
        s += int(curr["ROA"] > prev["ROA"])
        s += int(cfo > curr["NetIncomeLoss"])
        s += int((curr["TotalDebt"]/curr["Assets"]) < (prev["TotalDebt"]/prev["Assets"]))
        s += int(curr["CurrentRatio"] > prev["CurrentRatio"])
        s += int(curr["CommonStockSharesOutstanding"] <= prev["CommonStockSharesOutstanding"])
        s += int(curr["GrossMargin"] > prev["GrossMargin"])
        s += int(curr["AssetTurnover"] > prev["AssetTurnover"])
        scores.append(s)
    return pd.Series(scores, index=df.index, name="Piotroski_F")

# Apply to target
target_mcap = market_data[TARGET_TICKER]["market_cap"]
mcap_series = pd.Series([target_mcap] * len(target_ratios), index=target_ratios.index)
z = altman_z_score(target_ratios, mcap_series)
target_ratios["Z_score"], target_ratios["Z_zone"] = z["Z_score"], z["Zone"]
target_ratios["Piotroski_F"] = piotroski_f_score(target_ratios)

# Apply to all comps for comparison
health_rows = []
for t in ALL_TICKERS:
    r = compute_ratios(all_panels[t])
    mc = market_data[t]["market_cap"]
    zz = altman_z_score(r, pd.Series([mc] * len(r), index=r.index))
    ff = piotroski_f_score(r)
    health_rows.append({"ticker": t, "Z_score": zz["Z_score"].iloc[-1], "Piotroski_F": ff.iloc[-1]})

health_df = pd.DataFrame(health_rows)
fig = px.bar(health_df, x="ticker", y="Z_score", color="ticker",
             title="Altman Z-Score — Target vs. Comps (Latest FY)")
fig.show()
health_df

,ticker,Z_score,Piotroski_F
0,EPAM,4.432,5.000
1,ACN,3.518,4.000
2,CTSH,4.926,6.000
3,DXC,1.126,6.000
4,IBM,3.278,6.000


In [38]:
def compute_comps_multiples(ticker):
    md = market_data[ticker]
    r = compute_ratios(all_panels[ticker]).iloc[-1]
    ev = md["market_cap"] + md["total_debt"] - md["total_cash"]
    return {
        "ticker": ticker,
        "EV": ev,
        "EV_EBITDA": ev / r["EBITDA"] if r["EBITDA"] else np.nan,
        "EV_Revenue": ev / r["Revenues"] if r["Revenues"] else np.nan,
        "PE": md["trailing_pe"],
    }

comps_multiples = pd.DataFrame([compute_comps_multiples(t) for t in COMPS_TICKERS])

# P/E is "Not Meaningful" for companies with depressed/distorted earnings
# (e.g. DXC's 82.85x reflects a temporarily small net income denominator,
# not genuine valuation richness) — standard sell-side practice is to flag
# such multiples as N.M. and exclude them from the P/E median specifically,
# while still using the company's EV/EBITDA and EV/Revenue data.
PE_OUTLIER_THRESHOLD = 40  # flag anything above this as N.M.
comps_multiples["PE_flag"] = np.where(comps_multiples["PE"] > PE_OUTLIER_THRESHOLD, "N.M.", "")

median_ev_ebitda = comps_multiples["EV_EBITDA"].median()
median_ev_rev = comps_multiples["EV_Revenue"].median()
median_pe = comps_multiples.loc[comps_multiples["PE_flag"] != "N.M.", "PE"].median()

target_latest = compute_ratios(all_panels[TARGET_TICKER]).iloc[-1]
implied_ev_from_ebitda = median_ev_ebitda * target_latest["EBITDA"]
implied_ev_from_rev = median_ev_rev * target_latest["Revenues"]
implied_equity_from_pe = median_pe * target_latest["NetIncomeLoss"]

print(f"Comps median EV/EBITDA: {median_ev_ebitda:.1f}x  ->  Implied EV: ${implied_ev_from_ebitda:,.0f}")
print(f"Comps median EV/Revenue: {median_ev_rev:.2f}x  ->  Implied EV: ${implied_ev_from_rev:,.0f}")
print(f"Comps median P/E (excl. N.M.): {median_pe:.1f}x  ->  Implied Equity Value: ${implied_equity_from_pe:,.0f}")
print(f"\nExcluded from P/E median: {comps_multiples.loc[comps_multiples['PE_flag']=='N.M.', 'ticker'].tolist()}")

fig = px.box(comps_multiples.melt(id_vars=["ticker","PE_flag"], value_vars=["EV_EBITDA", "EV_Revenue", "PE"]),
             x="variable", y="value", points="all", title="Comps Multiple Distribution (P/E N.M. excluded from median)")
fig.show()
comps_multiples

Comps median EV/EBITDA: 6.1x  ->  Implied EV: $3,163,703,978
Comps median EV/Revenue: 0.99x  ->  Implied EV: $5,409,021,913
Comps median P/E (excl. N.M.): 9.9x  ->  Implied Equity Value: $3,735,150,820

Excluded from P/E median: ['DXC']


,ticker,EV,EV_EBITDA,EV_Revenue,PE,PE_flag
0,ACN,73987869696,7.236,1.062,9.890,
1,CTSH,19429174272,4.933,0.920,9.105,
2,DXC,3852835072,1.790,0.305,82.850,N.M.
3,IBM,293845626880,28.451,4.351,22.185,


In [39]:
def get_risk_free_rate():
    """Pulls the latest 10-Year Treasury yield from FRED's public CSV endpoint (no API key needed)."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10"
    df = pd.read_csv(url)
    df.columns = ["date", "rate"]
    df = df[df["rate"] != "."]
    df["rate"] = df["rate"].astype(float)
    return df["rate"].iloc[-1] / 100

RISK_FREE_RATE = get_risk_free_rate()
EQUITY_RISK_PREMIUM = 0.045   # standard long-run US ERP assumption — adjust as needed
TAX_RATE = 0.21

def compute_wacc(ticker):
    md = market_data[ticker]
    beta = md["beta"]
    cost_of_equity = RISK_FREE_RATE + beta * EQUITY_RISK_PREMIUM
    market_cap, total_debt = md["market_cap"], md["total_debt"]
    total_value = market_cap + total_debt
    cost_of_debt = RISK_FREE_RATE + 0.015  # simplified credit-spread assumption
    wacc = (market_cap/total_value)*cost_of_equity + (total_debt/total_value)*cost_of_debt*(1 - TAX_RATE)
    return wacc, cost_of_equity, cost_of_debt

WACC, COST_OF_EQUITY, COST_OF_DEBT = compute_wacc(TARGET_TICKER)
print(f"Risk-Free Rate: {RISK_FREE_RATE:.2%}")
print(f"Cost of Equity: {COST_OF_EQUITY:.2%} | Cost of Debt: {COST_OF_DEBT:.2%} | WACC: {WACC:.2%}")

Risk-Free Rate: 4.49%
Cost of Equity: 10.79% | Cost of Debt: 5.99% | WACC: 10.38%


In [40]:
DCF_ASSUMPTIONS = {
    "revenue_growth": [0.10, 0.09, 0.08, 0.07, 0.06],   # Yr1..Yr5 — EDIT based on your own research
    # Margin assumption: EPAM's operating margin has fallen 5 straight years
    # (14.4% -> 9.5%, per Cell 6). Rather than flat-lining at today's margin
    # (too optimistic — ignores a real structural trend) or extrapolating the
    # decline linearly for 5 more years (too pessimistic — management would
    # act on cost/pricing before margins kept falling indefinitely), this
    # assumes partial continuation of the pressure for 1-2 more years,
    # then stabilization/modest recovery as management responds.
    "ebit_margin": [0.090, 0.087, 0.087, 0.088, 0.090],
    "tax_rate": TAX_RATE,
    "da_pct_revenue": 0.03,
    "capex_pct_revenue": 0.035,
    "nwc_pct_revenue_change": 0.01,
    "terminal_growth": 0.025,
}

def run_dcf(base_revenue, assumptions, wacc):
    fcffs = []
    rev = base_revenue
    for g, margin in zip(assumptions["revenue_growth"], assumptions["ebit_margin"]):
        rev = rev * (1 + g)
        ebit = rev * margin
        da = rev * assumptions["da_pct_revenue"]
        capex = rev * assumptions["capex_pct_revenue"]
        d_nwc = rev * assumptions["nwc_pct_revenue_change"]
        nopat = ebit * (1 - assumptions["tax_rate"])
        fcffs.append(nopat + da - capex - d_nwc)

    years = np.arange(1, len(fcffs) + 1)
    disc_factors = 1 / (1 + wacc) ** years
    pv_fcff = np.array(fcffs) * disc_factors
    tv = fcffs[-1] * (1 + assumptions["terminal_growth"]) / (wacc - assumptions["terminal_growth"])
    pv_tv = tv * disc_factors[-1]
    return {"fcffs": fcffs, "pv_fcff": pv_fcff, "pv_terminal": pv_tv,
            "enterprise_value": pv_fcff.sum() + pv_tv}

dcf_result = run_dcf(target_latest["Revenues"], DCF_ASSUMPTIONS, WACC)
dcf_ev = dcf_result["enterprise_value"]
target_net_debt = market_data[TARGET_TICKER]["total_debt"] - market_data[TARGET_TICKER]["total_cash"]
dcf_equity_value = dcf_ev - target_net_debt

print(f"Projected FCFF (Yr1-5): {[round(x,1) for x in dcf_result['fcffs']]}")
print(f"DCF Enterprise Value: ${dcf_ev:,.0f}")
print(f"DCF Implied Equity Value: ${dcf_equity_value:,.0f}")

Projected FCFF (Yr1-5): [np.float64(336754925.8), np.float64(351555935.0), np.float64(379680409.8), np.float64(412231309.5), np.float64(449628522.6)]
DCF Enterprise Value: $5,000,746,746
DCF Implied Equity Value: $5,749,762,714


In [41]:
wacc_range = np.linspace(WACC - 0.02, WACC + 0.02, 5)
growth_range = np.linspace(0.015, 0.035, 5)

sensitivity = pd.DataFrame(
    index=[f"{w:.2%}" for w in wacc_range],
    columns=[f"{g:.2%}" for g in growth_range],
    dtype=float
)

for w in wacc_range:
    for g in growth_range:
        a = dict(DCF_ASSUMPTIONS)
        a["terminal_growth"] = g
        res = run_dcf(target_latest["Revenues"], a, w)
        sensitivity.loc[f"{w:.2%}", f"{g:.2%}"] = res["enterprise_value"]

fig = px.imshow(
    sensitivity, text_auto=".2s", aspect="auto",
    labels=dict(x="Terminal Growth", y="WACC", color="Enterprise Value ($)"),
    title=f"{TARGET_TICKER} — DCF Sensitivity: Enterprise Value"
)
fig.show()
sensitivity

,1.50%,2.00%,2.50%,3.00%,3.50%
8.38%,"5,948,076,291.333","6,319,910,299.254","6,755,034,806.080","7,271,112,155.814","7,893,050,727.660"
9.38%,"5,169,580,887.949","5,440,060,310.149","5,749,881,983.373","6,108,302,854.480","6,527,731,151.662"
10.38%,"4,567,288,112.499","4,771,078,422.389","5,000,746,746.056","5,261,556,382.100","5,560,301,762.340"
11.38%,"4,087,635,467.353","4,245,454,986.329","4,421,056,912.653","4,617,626,141.124","4,839,156,424.932"
12.38%,"3,696,763,230.921","3,821,709,730.059","3,959,308,992.130","4,111,585,451.264","4,281,019,750.211"


In [42]:
valuation_summary = pd.DataFrame({
    "Method": ["DCF", "Comps EV/EBITDA", "Comps EV/Revenue"],
    "Enterprise_Value": [dcf_ev, implied_ev_from_ebitda, implied_ev_from_rev]
})

spread_pct = (valuation_summary["Enterprise_Value"].max() - valuation_summary["Enterprise_Value"].min()) \
             / valuation_summary["Enterprise_Value"].median()

fig = px.bar(valuation_summary, x="Method", y="Enterprise_Value", color="Method",
             title=f"{TARGET_TICKER} — Valuation Cross-Validation (Spread: {spread_pct:.1%})")
fig.show()
print(f"Method spread: {spread_pct:.1%} {'⚠️ wide divergence — revisit assumptions' if spread_pct > 0.25 else '✓ reasonably converged'}")

Method spread: 44.9% ⚠️ wide divergence — revisit assumptions


In [43]:
report = f"""# {TARGET_TICKER} — Automated Valuation Report

## 1. Executive Summary
- DCF Implied Enterprise Value: ${dcf_ev:,.0f}
- Comps (EV/EBITDA) Implied EV: ${implied_ev_from_ebitda:,.0f}
- Comps (EV/Revenue) Implied EV: ${implied_ev_from_rev:,.0f}
- Cross-Method Spread: {spread_pct:.1%}
- Altman Z-Score: {target_ratios['Z_score'].iloc[-1]:.2f} ({target_ratios['Z_zone'].iloc[-1]})
- Piotroski F-Score: {target_ratios['Piotroski_F'].iloc[-1]:.0f}/9

## 2. Financial Trend Summary (5-Year)
{target_ratios[['fy','Revenues','GrossMargin','OperatingMargin','NetMargin','ROE']].to_markdown(index=False)}

## 3. DCF Assumptions
- WACC: {WACC:.2%}
- Terminal Growth: {DCF_ASSUMPTIONS['terminal_growth']:.2%}
- Revenue Growth (Yr1-5): {[f'{g:.1%}' for g in DCF_ASSUMPTIONS['revenue_growth']]}

## 4. Comps Set
{', '.join(COMPS_TICKERS)}

## 5. Recommendation
{"DCF and comps converge reasonably well, supporting confidence in the valuation range." if spread_pct <= 0.25 else "DCF and comps diverge meaningfully — recommend revisiting growth/margin/WACC assumptions or the comps set before finalizing a view."}
"""

output_path = f"{TARGET_TICKER}_valuation_report.md"
with open(output_path, "w") as f:
    f.write(report)

print(f"Report saved to {output_path}")
print(report)

Report saved to EPAM_valuation_report.md
# EPAM — Automated Valuation Report

## 1. Executive Summary
- DCF Implied Enterprise Value: $5,000,746,746
- Comps (EV/EBITDA) Implied EV: $3,163,703,978
- Comps (EV/Revenue) Implied EV: $5,409,021,913
- Cross-Method Spread: 44.9%
- Altman Z-Score: 4.43 (Safe)
- Piotroski F-Score: 5/9

## 2. Financial Trend Summary (5-Year)
|   fy |    Revenues |   GrossMargin |   OperatingMargin |   NetMargin |      ROE |
|-----:|------------:|--------------:|------------------:|------------:|---------:|
| 2021 | 3.75814e+09 |      0.339116 |          0.144304 |   0.128162  | 0.193659 |
| 2022 | 4.8247e+09  |      0.31878  |          0.118757 |   0.086931  | 0.139734 |
| 2023 | 4.69054e+09 |      0.305727 |          0.106862 |   0.08892   | 0.120166 |
| 2024 | 4.72794e+09 |      0.306781 |          0.115184 |   0.0961376 | 0.125243 |
| 2025 | 5.45706e+09 |      0.288346 |          0.09529  |   0.0692091 | 0.102707 |

## 3. DCF Assumptions
- WACC: 10.38%
- Term